# Notebook 10 — Vision Transformer (ViT) for Image Classification

## Learning Objectives
- Understand how ViT splits an image into patches and treats them as tokens.
- Build and train `VisionTransformer` on Fashion-MNIST.
- Visualise the patch grid — the core ViT preprocessing step.
- Compare ViT accuracy and parameter count against the CNN baseline from Notebook 09.
- Examine how the CLS token summarises the full image for classification.
- Identify failure cases and why ViT needs more data/epochs than CNNs.

## Big Picture
Dosovitskiy et al. (2020) showed that a pure Transformer applied to image patches can match or
exceed CNN performance when trained on sufficient data. The key insight: treat each image patch
as a word token and let global self-attention handle long-range spatial dependencies.

## Dataset
**Fashion-MNIST** — same split as Notebook 09 for a fair comparison.

## Architecture
```
Input [B, 1, 28, 28]
  └─ PatchEmbedding (patch_size=7): split into 4×4=16 patches
     each patch: 7×7×1=49 pixels → Linear(49→64) → [B, 16, 64]
  └─ Prepend CLS token                             → [B, 17, 64]
  └─ + LearnedPositionalEncoding                  → [B, 17, 64]
  └─ TransformerEncoder (2 blocks, 4 heads)       → [B, 17, 64]
  └─ Take CLS token output                        → [B, 64]
  └─ LayerNorm + Linear(64→10)                    → logits [B, 10]
```

## Theory
**Patch embedding** is the ViT analogue of word embedding: a learnable linear projection maps each
flat patch vector to a $d_{model}$-dimensional space.

**CLS token** is a special learnable vector prepended to the patch sequence. After encoding,
its output attends to all patches and serves as the image representation.

**Self-attention** lets every patch directly compare itself to every other patch in $O(1)$ layers,
capturing long-range spatial relationships that CNNs can only build up layer by layer.

**No inductive bias**: ViT treats patches as an unordered set (positions are added via learned PE).
This makes it more general but requires more data to learn spatial structure from scratch.

**Reference**: Dosovitskiy et al., *An Image is Worth 16×16 Words*, ICLR 2021.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary
from src.utils.paths import RUNS_VIT_CLS_DIR
from src.data.vision_datasets import load_fashion_mnist, FASHION_MNIST_CLASSES
from src.models.vit import VisionTransformer, PatchEmbedding
from src.visualization.plots import show_image_grid, plot_training_curves, plot_confusion_matrix
from src.visualization.patches import show_patches, show_patch_grid

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_VIT_CLS_DIR, clean=False)
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')

## Dataset
### 2. Load Fashion-MNIST

In [ ]:
BATCH_SIZE  = 64
NUM_EPOCHS  = 5
LR          = 1e-3
IMAGE_SIZE  = 28
PATCH_SIZE  = 7      # 28/7 = 4 patches per side → 16 patches total
D_MODEL     = 64
NUM_HEADS   = 4
NUM_LAYERS  = 2
DIM_FF      = 128
NUM_CLASSES = 10
IN_CHANNELS = 1      # greyscale

train_loader, val_loader, test_loader = load_fashion_mnist(
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

images, labels = next(iter(train_loader))
print(f'Image batch shape : {images.shape}  (B, C, H, W)')
print(f'Num patches per image: {(IMAGE_SIZE // PATCH_SIZE)**2}  ({IMAGE_SIZE//PATCH_SIZE}x{IMAGE_SIZE//PATCH_SIZE} grid)')
print(f'Pixels per patch     : {IN_CHANNELS * PATCH_SIZE * PATCH_SIZE}')

In [ ]:
# Visualise patches on one sample image
sample_img = images[0].numpy()  # [1, 28, 28]
# Unnormalise for display
sample_img_display = (sample_img * 0.353 + 0.286).clip(0, 1).squeeze(0)  # [28, 28]

show_patches(
    sample_img_display,
    patch_size=PATCH_SIZE,
    save_path=run_dir / 'patch_grid.png',
    title=f'Fashion-MNIST — Patch Grid (patch_size={PATCH_SIZE}, label={FASHION_MNIST_CLASSES[labels[0]]})',
)
print('Patch grid saved.')

# Show all 16 individual patches
show_patch_grid(
    sample_img_display,
    patch_size=PATCH_SIZE,
    save_path=run_dir / 'patch_grid_individual.png',
    title=f'Individual Patches — {FASHION_MNIST_CLASSES[labels[0]]}',
)
print('Individual patch grid saved.')

### 3. Build the Vision Transformer

In [ ]:
model = VisionTransformer(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=0.1,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters : {n_params:,}')

# Shape trace
x_sample = images[:4].to(device)   # [4, 1, 28, 28]
with torch.no_grad():
    logits = model(x_sample)        # [4, 10]

print(f'Input  shape : {x_sample.shape}   (batch, channels, H, W)')
print(f'Output shape : {logits.shape}   (batch, num_classes)')
print(f'Num patches  : {model.patch_embedding.num_patches}  (+1 CLS = {model.patch_embedding.num_patches+1} tokens)')

## Sanity Checks

In [ ]:
# 1. Output shape
assert logits.shape == (4, NUM_CLASSES), f'Unexpected shape: {logits.shape}'
print('Output shape check passed.')

# 2. Patch embedding shape check
patch_emb = PatchEmbedding(IMAGE_SIZE, PATCH_SIZE, IN_CHANNELS, D_MODEL)
patch_out = patch_emb(images[:2])
assert patch_out.shape == (2, (IMAGE_SIZE//PATCH_SIZE)**2, D_MODEL)
print(f'PatchEmbedding output : {patch_out.shape}  (batch, num_patches, d_model)')

# 3. return_patch_tokens
with torch.no_grad():
    logits2, patch_tokens = model(x_sample, return_patch_tokens=True)
assert patch_tokens.shape == (4, model.patch_embedding.num_patches, D_MODEL)
print(f'Patch tokens shape   : {patch_tokens.shape}  (batch, num_patches, d_model)')

# 4. No NaN
assert not torch.isnan(logits).any()
print('No NaN in output.')

print('All sanity checks passed!')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)                         # [B, 10]
        loss = F.cross_entropy(out, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    train_losses.append(t_loss / n_batches)

    # ── Validation ──────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            v_loss += F.cross_entropy(out, lbls).item()
            v_correct += (out.argmax(-1) == lbls).sum().item()
            v_total += lbls.size(0)
    val_losses.append(v_loss / len(val_loader))
    val_accs.append(v_correct / v_total)
    scheduler.step()

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train_loss={train_losses[-1]:.4f}  '
          f'val_loss={val_losses[-1]:.4f}  '
          f'val_acc={val_accs[-1]:.4f}')

print('Training complete!')

## Evaluation

In [ ]:
# Test set evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        out = model(imgs.to(device))
        all_preds.append(out.argmax(-1).cpu())
        all_labels.append(lbls)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = (all_preds == all_labels).mean()
print(f'ViT Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

# Load CNN results for comparison
import json
cnn_metrics_path = run_dir / 'cnn_metrics.json'
if cnn_metrics_path.exists():
    with open(cnn_metrics_path) as f:
        cnn_m = json.load(f)
    print(f'CNN Test accuracy : {cnn_m["test_accuracy"]:.4f}  ({cnn_m["test_accuracy"]*100:.2f}%)')
    print(f'CNN params        : {cnn_m["num_params"]:,}')
    print(f'ViT params        : {n_params:,}')
else:
    print('CNN metrics not found — run Notebook 09 first for comparison.')

print('\nPer-class report:')
print(classification_report(all_labels, all_preds, target_names=FASHION_MNIST_CLASSES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plot_confusion_matrix(
    cm,
    class_names=FASHION_MNIST_CLASSES,
    save_path=run_dir / 'vit_confusion_matrix.png',
    title='ViT — Confusion Matrix (Fashion-MNIST Test Set)',
)
print('ViT confusion matrix saved.')

# Save metrics
metrics = {
    'test_accuracy': float(test_acc),
    'final_val_accuracy': float(val_accs[-1]),
    'final_train_loss': float(train_losses[-1]),
    'final_val_loss': float(val_losses[-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'patch_size': PATCH_SIZE,
    'd_model': D_MODEL,
}
save_metrics_json(metrics, run_dir / 'vit_metrics.json')
print(f'Metrics saved to: {run_dir}/vit_metrics.json')

## Visualization

In [ ]:
# Training curves — ViT
history = {
    'train_loss': train_losses,
    'val_loss': val_losses,
    'val_accuracy': val_accs,
}
plot_training_curves(
    history,
    save_path=run_dir / 'vit_training_curve.png',
    title='ViT Training Curves (Fashion-MNIST)',
)
print('ViT training curve saved.')

In [ ]:
# CNN vs ViT accuracy comparison bar chart
fig, ax = plt.subplots(figsize=(7, 4))
models_names = ['SimpleCNN\n(3 epochs)', 'ViT\n(5 epochs)']
accuracies    = [cnn_m['test_accuracy'] if cnn_metrics_path.exists() else 0.0, test_acc]
params_k      = [cnn_m['num_params'] // 1000 if cnn_metrics_path.exists() else 0, n_params // 1000]
colors = ['steelblue', 'tomato']

bars = ax.bar(models_names, [a * 100 for a in accuracies], color=colors, width=0.4, edgecolor='black', linewidth=0.8)
for bar, acc, pk in zip(bars, accuracies, params_k):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc*100:.1f}%\n({pk}K params)', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0, 100)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CNN vs ViT — Fashion-MNIST Test Accuracy')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(run_dir / 'cnn_vs_vit_comparison.png', dpi=120)
plt.close(fig)
print('Comparison chart saved.')

In [ ]:
# Visualise patch token activations for one image
model.eval()
sample = images[:1].to(device)  # [1, 1, 28, 28]
with torch.no_grad():
    _, patch_toks = model(sample, return_patch_tokens=True)  # [1, 16, 64]

# Visualise L2 norm of each patch token (proxy for activation strength)
patch_norms = patch_toks[0].norm(dim=-1).cpu().numpy()   # [16]
patch_norms_grid = patch_norms.reshape(IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(sample[0, 0].cpu().numpy() * 0.353 + 0.286, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Input image: {FASHION_MNIST_CLASSES[labels[0]]}')
axes[0].axis('off')
im = axes[1].imshow(patch_norms_grid, cmap='hot', interpolation='nearest')
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Patch token L2 norms after Transformer')
axes[1].axis('off')
plt.suptitle('ViT Patch Token Activations')
plt.tight_layout()
fig.savefig(run_dir / 'vit_patch_activations.png', dpi=120)
plt.close(fig)
print('Patch activation map saved.')

## Failure Cases

**Why ViT may underperform CNN on small datasets like Fashion-MNIST:**

1. **Lack of inductive bias**: CNNs have translation equivariance built in. ViT must learn that
   nearby patches are correlated entirely from data — requiring more samples.

2. **Small image size**: With 28×28 and patch_size=7, we get only 16 tokens. The Transformer's
   global attention has little to gain from such a small sequence. For tiny images, a CNN's local
   receptive field is actually a better fit.

3. **Overfitting risk**: With only ~54K training images and no pre-training, a Transformer can
   overfit if d_model or num_layers is too large.

4. **Positional encoding on a 4×4 grid**: The learned positional encoding must assign distinct,
   meaningful embeddings to each of the 16 patch positions. With limited data this is harder than
   CNN's natural spatial hierarchy.

**In practice**: ViT wins when pre-trained on large datasets (ImageNet-21K, JFT-300M) then
fine-tuned. For small from-scratch tasks, CNNs are usually more efficient.

## Exercises

1. **Larger patch size**: Change `patch_size` to 14 (2 patches per side → 4 tokens). How does
   accuracy change? What is the tradeoff between patch granularity and token sequence length?

2. **Sinusoidal vs learned PE**: Replace `LearnedPositionalEncoding` with
   `SinusoidalPositionalEncoding` from `src.models.positional_encoding`. Does the fixed
   encoding hurt performance?

3. **Without CLS token**: Remove the CLS token and use global average pooling over patch tokens
   instead. Modify the `forward` method and retrain. Compare accuracy.

4. **Attention head visualisation**: Extract the attention weights from the first encoder block's
   first head for a single image. Display the attention map as a 4×4 grid. Which patches attend
   to each other most strongly?

5. **Scaling experiment**: Train ViT with d_model=128 and num_layers=4. How many parameters does
   it have? Does the larger model converge to better accuracy?

## Key Takeaways

- **Patch embedding** is the bridge between pixels and tokens: each patch is linearly projected to d_model.
- **CLS token** aggregates information from all patches via attention and represents the full image.
- **Global self-attention** captures long-range dependencies between patches — impossible for a small
  CNN receptive field in early layers.
- **No inductive bias** is a double-edged sword: ViT is flexible but requires more data or pre-training
  to learn spatial structure.
- On small datasets, CNN inductive biases (locality, translation equivariance) are a real advantage.
- **ViT excels when**: (1) large training data, (2) large model capacity, (3) high-resolution images
  where global context matters.

## Saved Outputs

In [ ]:
save_markdown_report(
    title='Vision Transformer — Fashion-MNIST Classification',
    summary=(
        f'ViT (image_size=28, patch_size=7, d_model=64) trained for {NUM_EPOCHS} epochs. '
        f'Test accuracy: {test_acc*100:.2f}%. {n_params:,} trainable parameters.'
    ),
    metrics=metrics,
    figures=['vit_training_curve.png', 'vit_confusion_matrix.png',
             'patch_grid.png', 'cnn_vs_vit_comparison.png'],
    tables=[],
    output_path=run_dir / 'session_report.md',
    device=str(device),
    hyperparams={
        'patch_size': PATCH_SIZE,
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'batch_size': BATCH_SIZE,
        'epochs': NUM_EPOCHS,
        'lr': LR,
    },
    architecture='PatchEmbedding(7→64) + CLS + LearnedPE + TransformerEncoder×2 + CLS→Linear(64→10)',
    loss_fn='CrossEntropyLoss',
)

update_runs_summary(
    session_name='vit_classification',
    report_path=run_dir / 'session_report.md',
    metrics=metrics,
    figures=['vit_training_curve.png', 'cnn_vs_vit_comparison.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/vit_metrics.json')
print(f'  {run_dir}/vit_training_curve.png')
print(f'  {run_dir}/patch_grid.png')
print(f'  {run_dir}/session_report.md')